# OpenAI Function Calling

**Goal:** Connect LLMs with external tools and APIs.

In this notebook, the model analyzes the user's request, decides whether one of the available tools should be used, generates a structured function call, and uses the returned data to produce the final response.

API: Frankfurter API

Workflow:

User Request -> OpenAI Model (generates function call) -> Python executes the function -> External API returns data -> OpenAI Model (generates final response)

In [1]:
from openai import OpenAI
from dotenv import load_dotenv

import json
import os
import requests

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## Define a Python Function

Create a Python function that retrieves the latest exchange rate from the Frankfurter API. This function will be executed when requested by the model.
```

In [2]:
def get_exchange_rate(base: str, target: str):
    url = f"https://api.frankfurter.dev/v1/latest?base={base}&symbols={target}"

    response = requests.get(url)
    response.raise_for_status()

    data = response.json()

    return {
        "base": base,
        "target": target,
        "rate": data["rates"][target]
    }

## Register the Tool

Define the function schema and make it available to the model. The schema describes what the function does and which arguments it expects.
```

In [3]:
tools = [
    {
        "type": "function",
        "name": "get_exchange_rate",
        "description": "Get the latest currency exchange rate.",
        "parameters": {
            "type": "object",
            "properties": {
                "base": {
                    "type": "string",
                    "description": "Base currency code."
                },
                "target": {
                    "type": "string",
                    "description": "Target currency code."
                }
            },
            "required": [
                "base",
                "target"
            ],
            "additionalProperties": False
        }
    }
]

## Generate a Function Call

Send a user request along with the available tools. If the model determines that a tool is needed, it returns a structured function call instead of a direct answer.

In [17]:
response = client.responses.create(
    model="gpt-4.1-mini",
    input="Я лечу в Японию. У меня бюджет 2000 евро. Сколько это сейчас в иенах?",
    tools=tools
)

In [18]:
response.output

[ResponseFunctionToolCall(arguments='{"base":"EUR","target":"JPY"}', call_id='call_DkNde7THM4ZONUnunR7XoAGn', name='get_exchange_rate', type='function_call', id='fc_0469c9fdd2a5b22e006a3e4ead5c9881a383b095dbd801b9bd', namespace=None, status='completed')]

In [19]:
tool_call = response.output[0]

arguments = json.loads(tool_call.arguments)

result = get_exchange_rate(
    base=arguments["base"],
    target=arguments["target"]
)

print(result)

{'base': 'EUR', 'target': 'JPY', 'rate': 183.57}


## Return the Function Result

Send the function output back to the model. Using the previous response, the model incorporates the returned data and generates the final answer for the user.

In [20]:
response = client.responses.create(
    model="gpt-4.1-mini",
    previous_response_id=response.id,
    input=[
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(result)
        }
    ]
)

In [21]:
print(response.output_text)

Сейчас 1 евро примерно равен 183.57 иенам. Значит, 2000 евро — это примерно 367,140 иен (2000 × 183.57).  
Желаю хорошего путешествия в Японию!
